# LeetCode 812 — Largest Triangle Area bằng Python

**Link đề:** https://leetcode.com/problems/largest-triangle-area/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Geometry (hình học), Polygons

```python
class Solution:
    def largestTriangleArea(self, points: list[list[int]]) -> float:
        ...
```

## Mục tiêu học
1. Tính diện tích tam giác từ **tọa độ** 3 đỉnh bằng **công thức Shoelace / tích có hướng (cross product)**.
2. So sánh với công thức **Heron** (dùng độ dài cạnh) và hiểu vì sao Shoelace tốt hơn (chính xác, không căn bậc hai).
3. Liệt kê mọi bộ 3 điểm bằng 3 vòng lặp và bằng `itertools.combinations`.
4. (Nâng cao) Nhận xét tam giác lớn nhất có đỉnh trên **bao lồi** (convex hull) để giảm số điểm cần xét.
5. So sánh số thực bằng sai số (`math.isclose`), trace, chứng minh, Time/Space.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng `points` các điểm trên mặt phẳng `points[i] = [xi, yi]`. Trả về **diện tích tam giác lớn nhất** tạo được từ **ba điểm khác nhau** bất kỳ. Đáp án sai lệch trong `10^-5` được chấp nhận.

**Ví dụ 1:** `points = [[0,0],[0,1],[1,0],[0,2],[2,0]]` → `2.00000`
(tam giác `(0,0), (0,2), (2,0)`: vuông cân cạnh 2, diện tích `2·2/2 = 2`).

**Ví dụ 2:** `points = [[1,0],[0,0],[0,1]]` → `0.50000`

**Constraints:** `3 <= points.length <= 50`; `-50 <= xi, yi <= 50`; các điểm **đôi một khác nhau**.

Lưu ý: ba điểm **thẳng hàng** tạo "tam giác" diện tích 0 — vẫn hợp lệ, chỉ là không lớn nhất.

In [1]:
points1 = [[0, 0], [0, 1], [1, 0], [0, 2], [2, 0]]
points2 = [[1, 0], [0, 0], [0, 1]]
# Vẽ "bản đồ" chữ của ví dụ 1 (y từ trên xuống)
grid_pts = {tuple(p) for p in points1}
for y in range(2, -1, -1):
    print(f"y={y} " + " ".join("●" if (x, y) in grid_pts else "·" for x in range(3)))
print("     x=0 1 2")

y=2 ● · ·
y=1 ● · ·
y=0 ● ● ●
     x=0 1 2


`tuple(p)` đổi list `[0, 2]` thành tuple `(0, 2)` — tuple **bất biến** nên dùng được làm phần tử của set (list thì không, vì list có thể thay đổi).

## 2. Nền tảng toán: diện tích tam giác từ tọa độ

### 2.1. Công thức "đáy × cao / 2" chỉ tiện khi có cạnh song song trục
Với `(0,0), (0,2), (2,0)`: đáy 2 (trục x), cao 2 → `2`. Nhưng với tam giác "xiên" thì khó tìm chiều cao.

### 2.2. Tích có hướng (cross product) 2D
Với hai vector `u = (ux, uy)`, `v = (vx, vy)`: `cross(u, v) = ux·vy − uy·vx`.

`|cross(u, v)|` = diện tích **hình bình hành** dựng trên `u`, `v` → diện tích tam giác = **một nửa**.

Với 3 điểm `A, B, C`: `u = B − A`, `v = C − A`:

`S = |(xB − xA)(yC − yA) − (yB − yA)(xC − xA)| / 2`

Dấu của `cross` cho biết chiều: dương = ngược chiều kim đồng hồ (rẽ trái), âm = cùng chiều, `0` = thẳng hàng.

In [2]:
def cross(o, a, b):
    # tích có hướng của OA và OB
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

A, B_, C = (0, 0), (2, 0), (0, 2)
print("cross(A, B, C) =", cross(A, B_, C), "-> diện tích =", abs(cross(A, B_, C)) / 2)
print("cross(A, C, B) =", cross(A, C, B_), "(đổi thứ tự -> đổi dấu, |.| không đổi)")
print("thẳng hàng:", cross((0, 0), (1, 1), (3, 3)))

cross(A, B, C) = 4 -> diện tích = 2.0
cross(A, C, B) = -4 (đổi thứ tự -> đổi dấu, |.| không đổi)
thẳng hàng: 0


### 2.3. Công thức Shoelace (dây giày) — dạng triển khai

Khai triển cross product ra ta được dạng đối xứng hay gặp:

`S = |x1(y2 − y3) + x2(y3 − y1) + x3(y1 − y2)| / 2`

Kiểm tra bằng tay với `(1,0), (0,0), (0,1)`: `1·(0 − 1) + 0·(1 − 0) + 0·(0 − 0) = −1` → `|−1| / 2 = 0.5` ✔.

In [3]:
def area_shoelace(p, q, r) -> float:
    (x1, y1), (x2, y2), (x3, y3) = p, q, r          # "giải nén" (unpacking) lồng nhau
    return abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2)) / 2

print(area_shoelace([1, 0], [0, 0], [0, 1]))
print(area_shoelace([0, 0], [0, 2], [2, 0]))
print(area_shoelace([0, 0], [1, 1], [3, 3]))     # thẳng hàng
print(area_shoelace([-3, 1], [4, -2], [1, 5]))   # tam giác xiên

0.5
2.0
0.0
20.0


### 2.4. Unpacking `(x1, y1), (x2, y2), (x3, y3) = p, q, r`

Python cho phép gán nhiều biến cùng lúc theo **cấu trúc**: `a, b = [3, 4]` gán `a=3, b=4`. Lồng thêm ngoặc để tách từng điểm thành `x`, `y`.

In [4]:
a, b = [3, 4]
print(a, b)
(x1, y1), (x2, y2) = [0, 5], [7, 8]
print(x1, y1, x2, y2)

3 4
0 5 7 8


### 2.5. Công thức Heron (để so sánh)

Với độ dài 3 cạnh `a, b, c` và nửa chu vi `s = (a+b+c)/2`: `S = √(s(s−a)(s−b)(s−c))`.

Cần `math.dist` (khoảng cách Euclid) và `math.sqrt`. Nhược điểm: dùng căn bậc hai → **sai số làm tròn**; với tam giác gần thẳng hàng biểu thức trong căn có thể ra **số âm rất nhỏ** → lỗi `math domain error`.

In [5]:
import math

def area_heron(p, q, r) -> float:
    a, b, c = math.dist(q, r), math.dist(p, r), math.dist(p, q)
    s = (a + b + c) / 2
    val = s * (s - a) * (s - b) * (s - c)
    return math.sqrt(max(val, 0.0))          # max(..., 0) để tránh âm do làm tròn

print("dist((0,0),(3,4)) =", math.dist((0, 0), (3, 4)))
print(area_heron([1, 0], [0, 0], [0, 1]), area_heron([-3, 1], [4, -2], [1, 5]))
a, b, c = math.dist((0, 0), (50, 49)), math.dist((0, 0), (49, 48)), math.dist((49, 48), (50, 49))
s = (a + b + c) / 2
print("thẳng hàng (0,0),(49,48),(50,49)? Heron trong căn =", s * (s - a) * (s - b) * (s - c),
      "| Shoelace =", area_shoelace([0, 0], [49, 48], [50, 49]))

dist((0,0),(3,4)) = 5.0
0.49999999999999983 19.999999999999996
thẳng hàng (0,0),(49,48),(50,49)? Heron trong căn = 0.2500000000543611 | Shoelace = 0.5


Chú ý: `(0,0), (49,48), (50,49)` **không** thẳng hàng (diện tích Shoelace = 0.5 — chính xác vì toàn số nguyên); Heron vẫn cho số dương nhưng bị nhiễu do làm tròn. Shoelace với tọa độ nguyên chỉ có **một** phép chia cho 2 → kết quả chính xác tuyệt đối.

## 3. Cách 1 — Brute force: thử mọi bộ 3 điểm (3 vòng lặp)

`n ≤ 50` → số bộ 3 là `C(50, 3) = 19 600` — rất nhỏ. Dùng `i < j < k` để mỗi bộ chỉ xét một lần.

In [6]:
def largest_triangle_bruteforce(points: list[list[int]]) -> float:
    n = len(points)
    best = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                best = max(best, area_shoelace(points[i], points[j], points[k]))
    return best

print(largest_triangle_bruteforce(points1), largest_triangle_bruteforce(points2))
print("C(50, 3) =", math.comb(50, 3))

2.0 0.5
C(50, 3) = 19600


### Vì sao `range(i + 1, n)`?

Bộ `{A, B, C}` và `{B, A, C}` là **cùng một tam giác**. Ép `i < j < k` giúp liệt kê mỗi tổ hợp đúng **một** lần (thay vì 6 lần với 3 vòng `range(n)` độc lập) và tự động tránh chọn trùng một điểm.

In [7]:
n = 4
triples = [(i, j, k) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)]
print(triples, "->", len(triples), "bộ = C(4,3) =", math.comb(4, 3))

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)] -> 4 bộ = C(4,3) = 4


## 4. Cách 2 — `itertools.combinations`: liệt kê tổ hợp gọn gàng

`combinations(seq, 3)` sinh mọi bộ 3 phần tử theo thứ tự xuất hiện, không lặp — đúng điều 3 vòng `i < j < k` làm.

In [8]:
from itertools import combinations
print(list(combinations("ABCD", 3)))

def largest_triangle(points: list[list[int]]) -> float:
    return max(area_shoelace(p, q, r) for p, q, r in combinations(points, 3))

print(largest_triangle(points1), largest_triangle(points2))

[('A', 'B', 'C'), ('A', 'B', 'D'), ('A', 'C', 'D'), ('B', 'C', 'D')]
2.0 0.5


## 5. Nút thắt và insight

Với constraints `n ≤ 50`, `O(n³)` là **đủ** — đây là lời giải chính được chấp nhận. "Nút thắt" thật sự của bài không phải tốc độ mà là **công thức diện tích**: dùng Shoelace (số nguyên, không căn) thay vì Heron.

Nếu `n` lớn (ví dụ `10^4`), `O(n³)` không còn chạy được. Khi đó dùng nhận xét hình học:

> Tam giác diện tích lớn nhất luôn có cả 3 đỉnh nằm trên **bao lồi** (convex hull) của tập điểm.

Trực giác: nếu một đỉnh nằm **bên trong**, ta có thể "đẩy" nó ra xa cạnh đối diện hơn (tới một điểm trên bao) → chiều cao tăng → diện tích tăng (hoặc giữ nguyên).

## 6. Lời giải chính — 3 vòng lặp + Shoelace (giảng từng đoạn)

```python
best = 0.0                                      # (1) chưa có tam giác nào
for i in range(n):                              # (2) chọn đỉnh thứ nhất
    for j in range(i + 1, n):                   # (3) đỉnh thứ hai, sau i
        for k in range(j + 1, n):               # (4) đỉnh thứ ba, sau j
            (x1,y1),(x2,y2),(x3,y3) = points[i], points[j], points[k]
            area2 = abs(x1*(y2-y3) + x2*(y3-y1) + x3*(y1-y2))   # (5) GẤP ĐÔI diện tích, số nguyên
            best = max(best, area2)             # (6) cập nhật max trên số nguyên
return best / 2                                 # (7) chia 2 một lần duy nhất ở cuối
```
Mẹo `(5)–(7)`: so sánh **gấp đôi diện tích** (số nguyên) và chỉ chia 2 khi trả kết quả → không có số thực trong vòng lặp.

In [9]:
def largest_triangle_int(points: list[list[int]]) -> float:
    n = len(points)
    best2 = 0
    for i in range(n):
        x1, y1 = points[i]
        for j in range(i + 1, n):
            x2, y2 = points[j]
            for k in range(j + 1, n):
                x3, y3 = points[k]
                area2 = abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
                if area2 > best2:
                    best2 = area2
    return best2 / 2

print(largest_triangle_int(points1), largest_triangle_int(points2))

2.0 0.5


## 7. TRACE từng bước

Ví dụ 1 có `C(5,3) = 10` bộ. In mọi bộ, gấp đôi diện tích, và khi nào `best` thay đổi:

| bộ (i,j,k) | điểm | 2S | best (2S) sau |
|---|---|---|---|
| (0,1,2) | (0,0)(0,1)(1,0) | 1 | 1 |
| (0,1,3) | (0,0)(0,1)(0,2) | 0 (thẳng hàng) | 1 |
| ... | ... | ... | ... |
| (0,3,4) | (0,0)(0,2)(2,0) | 4 | 4 |

In [10]:
def trace_largest_triangle(points):
    n, best2 = len(points), 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                (x1, y1), (x2, y2), (x3, y3) = points[i], points[j], points[k]
                area2 = abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
                note = ""
                if area2 > best2:
                    note = f"  <- best {best2} -> {area2}"
                    best2 = area2
                elif area2 == 0:
                    note = "  (thẳng hàng)"
                print(f"({i},{j},{k}) {points[i]}{points[j]}{points[k]}  2S={area2}{note}")
    print("Kết quả:", best2 / 2)
    return best2 / 2

_ = trace_largest_triangle(points1)

(0,1,2) [0, 0][0, 1][1, 0]  2S=1  <- best 0 -> 1
(0,1,3) [0, 0][0, 1][0, 2]  2S=0  (thẳng hàng)
(0,1,4) [0, 0][0, 1][2, 0]  2S=2  <- best 1 -> 2
(0,2,3) [0, 0][1, 0][0, 2]  2S=2
(0,2,4) [0, 0][1, 0][2, 0]  2S=0  (thẳng hàng)
(0,3,4) [0, 0][0, 2][2, 0]  2S=4  <- best 2 -> 4
(1,2,3) [0, 1][1, 0][0, 2]  2S=1
(1,2,4) [0, 1][1, 0][2, 0]  2S=1
(1,3,4) [0, 1][0, 2][2, 0]  2S=2
(2,3,4) [1, 0][0, 2][2, 0]  2S=2
Kết quả: 2.0


## 8. Code nộp LeetCode — lời giải chính

In [11]:
class Solution:
    def largestTriangleArea(self, points: list[list[int]]) -> float:
        n = len(points)
        best2 = 0
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                for k in range(j + 1, n):
                    x3, y3 = points[k]
                    best2 = max(best2, abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2)))
        return best2 / 2

sol = Solution()
assert abs(sol.largestTriangleArea(points1) - 2.0) < 1e-5
assert abs(sol.largestTriangleArea(points2) - 0.5) < 1e-5
print("Solution OK")

Solution OK


In [12]:
# Phiên bản ngắn gọn với combinations (cũng nộp được)
class SolutionShort:
    def largestTriangleArea(self, points: list[list[int]]) -> float:
        return max(abs((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))
                   for a, b, c in combinations(points, 3)) / 2

print(SolutionShort().largestTriangleArea(points1), SolutionShort().largestTriangleArea(points2))

2.0 0.5


## 9. Cách 4 (nâng cao) — chỉ xét các điểm trên bao lồi

**Bao lồi** (convex hull) là đa giác lồi nhỏ nhất bao mọi điểm — như căng dây thun quanh các đinh. Thuật toán **Andrew's monotone chain**:
1. Sắp xếp điểm theo `(x, y)`.
2. Dựng **nửa dưới**: duyệt trái → phải, giữ stack; nếu 2 điểm cuối stack và điểm mới **không rẽ trái** (`cross <= 0`) thì `pop`.
3. Dựng **nửa trên** tương tự, duyệt phải → trái.
4. Ghép hai nửa (bỏ điểm trùng ở đầu/cuối).

Sau đó brute force trên `h` điểm của bao: `O(n log n + h³)`. Trên lưới `[-50, 50]²` thì `h` nhỏ, nên rất nhanh khi `n` lớn.

In [13]:
def convex_hull(points):
    pts = sorted(map(tuple, points))
    if len(pts) <= 2:
        return pts
    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)
    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)
    return lower[:-1] + upper[:-1]

def largest_triangle_hull(points):
    hull = convex_hull(points)
    if len(hull) < 3:                 # mọi điểm thẳng hàng
        return 0.0
    return largest_triangle_int([list(p) for p in hull])

print("hull ví dụ 1:", convex_hull(points1))
print(largest_triangle_hull(points1), largest_triangle_hull(points2))

hull ví dụ 1: [(0, 0), (2, 0), (0, 2)]
2.0 0.5


Trong ví dụ 1, điểm `(0,1)` và `(1,0)` nằm **trên cạnh** của bao nên bị loại (`cross <= 0` loại cả điểm thẳng hàng) — chúng không thể tạo tam giác lớn hơn.

## 10. Vì sao đúng?

**Công thức:** `cross(B−A, C−A)` = diện tích có dấu của hình bình hành `ABDC`; tam giác là một nửa → `|cross|/2`. Shoelace chỉ là khai triển đại số của cross.

**Duyệt đủ:** ba vòng `i < j < k` sinh **mọi** tập 3 điểm khác nhau đúng một lần (song ánh với tổ hợp chập 3). `best` = max trên tất cả ⇒ đúng là diện tích lớn nhất.

**Gấp đôi diện tích:** `max(2S) / 2 = max(S)` vì nhân với hằng dương không đổi thứ tự.

**Bao lồi (phác thảo):** cố định 2 đỉnh `B, C`, diện tích tỉ lệ với khoảng cách từ đỉnh thứ 3 tới đường `BC` — một hàm tuyến tính (lấy trị tuyệt đối) theo tọa độ, nên đạt max tại một đỉnh của bao lồi. Lập luận lặp cho cả 3 đỉnh ⇒ tồn tại tam giác lớn nhất có cả 3 đỉnh trên bao.

## 11. Độ phức tạp và so sánh

`n` = số điểm (≤ 50), `h` = số điểm trên bao lồi.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. 3 vòng + Shoelace (float) | `O(n³)` | `O(1)` | Dễ hiểu |
| 2. `combinations` | `O(n³)` | `O(1)` (generator) | Gọn |
| 3. 3 vòng + gấp đôi diện tích nguyên | `O(n³)` | `O(1)` | **Khuyến nghị**, chính xác |
| 3'. Heron | `O(n³)` | `O(1)` | Sai số, có căn |
| 4. Bao lồi + brute force | `O(n log n + h³)` | `O(n)` | Khi `n` lớn |

In [14]:
import random, time
random.seed(812)
all_grid = [[x, y] for x in range(-50, 51) for y in range(-50, 51)]
big = random.sample(all_grid, 400)          # vượt constraints (n=400) để thấy khác biệt
for name, f in [("3 vòng (nguyên)", largest_triangle_int), ("bao lồi", largest_triangle_hull)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:<16} -> {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")
print("số điểm trên bao:", len(convex_hull(big)))

3 vòng (nguyên)  -> 4748.0  (2005.9 ms)
bao lồi          -> 4748.0  (0.5 ms)
số điểm trên bao: 15


## 12. Bộ test

So sánh số thực bằng **sai số** chứ không dùng `==` (dù ở đây kết quả luôn là bội của 0.5 nên `==` cũng an toàn): `math.isclose(a, b, abs_tol=1e-5)`.

1. Hai ví dụ đề.
2. Đúng 3 điểm.
3. Ba điểm thẳng hàng → `0.0`.
4. Tọa độ âm, tam giác xiên.
5. Bốn góc của hình vuông lớn nhất `[-50, 50]²` → `100·100/2 = 5000`.
6. Đối chiếu mọi cách (cả Heron) trên dữ liệu ngẫu nhiên.

In [15]:
def check(name, actual, expected):
    assert math.isclose(actual, expected, abs_tol=1e-5), (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", largest_triangle_int(points1), 2.0)
check("đề 2", largest_triangle_int(points2), 0.5)
check("thẳng hàng", largest_triangle_int([[0, 0], [1, 1], [2, 2]]), 0.0)
check("tam giác xiên", largest_triangle_int([[-3, 1], [4, -2], [1, 5]]), 20.0)
check("4 góc hình vuông", largest_triangle_int([[-50, -50], [-50, 50], [50, -50], [50, 50]]), 5000.0)
print("isclose(0.1+0.2, 0.3):", math.isclose(0.1 + 0.2, 0.3), "| 0.1+0.2 == 0.3:", 0.1 + 0.2 == 0.3)

PASS đề 1: 2.0
PASS đề 2: 0.5
PASS thẳng hàng: 0.0
PASS tam giác xiên: 20.0
PASS 4 góc hình vuông: 5000.0
isclose(0.1+0.2, 0.3): True | 0.1+0.2 == 0.3: False


In [16]:
for _ in range(300):
    pts = random.sample(all_grid, random.randint(3, 25))
    expected = largest_triangle_int(pts)
    assert math.isclose(largest_triangle_bruteforce(pts), expected)
    assert math.isclose(largest_triangle(pts), expected)
    assert math.isclose(Solution().largestTriangleArea(pts), expected)
    assert math.isclose(SolutionShort().largestTriangleArea(pts), expected)
    assert math.isclose(largest_triangle_hull(pts), expected)
    heron = max(area_heron(p, q, r) for p, q, r in combinations(pts, 3))
    assert math.isclose(heron, expected, abs_tol=1e-6)
print("300 bộ điểm ngẫu nhiên: 6 cách khớp nhau (Heron khớp trong sai số)")

300 bộ điểm ngẫu nhiên: 6 cách khớp nhau (Heron khớp trong sai số)


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên trị tuyệt đối
```python
area = (x1*(y2-y3) + x2*(y3-y1) + x3*(y1-y2)) / 2     # SAI: có thể âm
```
Tam giác đi theo chiều kim đồng hồ cho giá trị âm → `max` bỏ qua tam giác lớn.

### Lỗi 2: Quên chia 2
Trả về diện tích hình bình hành (gấp đôi).

### Lỗi 3: Dùng chia nguyên `// 2`
`1 // 2 = 0` → mất phần `.5` (ví dụ 2 ra `0` thay vì `0.5`).

### Lỗi 4: Khởi tạo `best = -1` hay `None` rồi trả về nguyên
Không sai thuật toán nhưng dễ trả kiểu sai; diện tích luôn ≥ 0 nên `0.0` là khởi tạo an toàn.

In [17]:
def wrong_no_abs(points):
    return max((p[0]*(q[1]-r[1]) + q[0]*(r[1]-p[1]) + r[0]*(p[1]-q[1])) / 2 for p, q, r in combinations(points, 3))

def wrong_floor_div(points):
    return max(abs(p[0]*(q[1]-r[1]) + q[0]*(r[1]-p[1]) + r[0]*(p[1]-q[1])) // 2 for p, q, r in combinations(points, 3))

cw = [[0, 0], [0, 4], [3, 0]]     # thứ tự này cho giá trị có dấu âm
print("không abs :", wrong_no_abs(cw), "| đúng:", largest_triangle_int(cw))
print("dùng //   :", wrong_floor_div(points2), "| đúng:", largest_triangle_int(points2))

không abs : -6.0 | đúng: 6.0
dùng //   : 0 | đúng: 0.5


## 14. Mẹo nhận dạng pattern

- "Diện tích / hướng rẽ / thẳng hàng" với **tọa độ nguyên** ⇒ **cross product** (Shoelace), tránh căn và lượng giác.
- `n ≤ 50` (hoặc ≤ 100) và hỏi về bộ 3 ⇒ `O(n³)` brute force là chấp nhận được.
- "Lớn nhất / xa nhất" trong tập điểm ⇒ nghĩ tới **bao lồi** khi `n` lớn.
- Kết quả là số thực ⇒ tính bằng **số nguyên** lâu nhất có thể, chia ở bước cuối; test bằng `isclose`.

### Biến thể

1. **Diện tích đa giác** bất kỳ (Shoelace tổng quát): `S = |Σ (x_i·y_{i+1} − x_{i+1}·y_i)| / 2`.
2. **Valid Boomerang (LeetCode 1037)**: 3 điểm có thẳng hàng không? → `cross != 0`.
3. **Max Points on a Line (149)**: nhiều điểm thẳng hàng nhất — dùng hệ số góc/cross.
4. **Erect the Fence (587)**: chính là bao lồi.
5. Đếm số tam giác **có diện tích dương** (không thẳng hàng).

In [18]:
# Biến thể 1: Shoelace cho đa giác (các đỉnh theo thứ tự quanh biên)
def polygon_area(vertices):
    n = len(vertices)
    s = 0
    for i in range(n):
        x1, y1 = vertices[i]
        x2, y2 = vertices[(i + 1) % n]      # % n: đỉnh cuối nối về đỉnh đầu
        s += x1 * y2 - x2 * y1
    return abs(s) / 2

print(polygon_area([[0, 0], [4, 0], [4, 3], [0, 3]]))   # hình chữ nhật 4x3 = 12
print(polygon_area([[0, 0], [2, 0], [0, 2]]))           # tam giác = 2

12.0
2.0


## 15. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (tính tay):** Tính diện tích tam giác `(1, 2), (4, 6), (7, 2)` bằng Shoelace. Gán `answer_ex1` (số thực).

**Bài 2 (điền chỗ trống):** Hoàn thiện `triangle_area_exercise(p, q, r)` bằng cross product.

**Bài 3 (biến thể):** `is_boomerang(points)` — 3 điểm cho trước **không** thẳng hàng (và khác nhau) thì `True`.

**Bài 4 (đếm):** `count_positive_triangles(points)` đếm số bộ 3 điểm tạo tam giác diện tích > 0.

**Bài 5 (mini challenge):** `largest_triangle_vertices(points)` trả về **bộ 3 điểm** (theo thứ tự xuất hiện) tạo tam giác lớn nhất (hòa thì lấy bộ gặp trước khi duyệt `combinations`).

In [19]:
answer_ex1 = None   # TODO

def triangle_area_exercise(p, q, r) -> float:
    # TODO: |(q-p) x (r-p)| / 2
    pass

def is_boomerang(points: list[list[int]]) -> bool:
    # TODO
    pass

# assert triangle_area_exercise([0, 0], [0, 2], [2, 0]) == 2.0
# assert is_boomerang([[1, 1], [2, 3], [3, 2]]) == True
# assert is_boomerang([[1, 1], [2, 2], [3, 3]]) == False

In [20]:
def count_positive_triangles(points: list[list[int]]) -> int:
    # TODO
    pass

def largest_triangle_vertices(points: list[list[int]]):
    # TODO: max(..., key=...)
    pass

# assert count_positive_triangles([[0, 0], [0, 1], [1, 0], [0, 2], [2, 0]]) == 8
# assert largest_triangle_vertices([[0, 0], [0, 1], [1, 0], [0, 2], [2, 0]]) == ([0, 0], [0, 2], [2, 0])

### Gợi ý 3 mức

**Bài 1** — (1) Áp `x1(y2−y3) + x2(y3−y1) + x3(y1−y2)`. (2) `1·(6−2) + 4·(2−2) + 7·(2−6)`. (3) Lấy trị tuyệt đối rồi chia 2 (đáy 6, cao 4 để kiểm tra).

**Bài 2** — (1) Hai vector từ `p`. (2) `ux, uy = q[0]-p[0], q[1]-p[1]`, tương tự `v`. (3) `abs(ux*vy - uy*vx) / 2`.

**Bài 3** — (1) Thẳng hàng ⇔ diện tích 0. (2) Điểm trùng cũng cho diện tích 0. (3) `return cross(...) != 0`.

**Bài 4** — (1) Duyệt `combinations`. (2) Đếm khi `2S > 0`. (3) Ví dụ 1 có 10 bộ, 2 bộ thẳng hàng (`(0,0),(0,1),(0,2)` và `(0,0),(1,0),(2,0)`).

**Bài 5** — (1) `max(combinations(points, 3), key=lambda t: area(*t))`. (2) `*t` "trải" tuple thành 3 đối số. (3) `max` giữ phần tử gặp trước khi hòa.

---

## 16. Đáp án tham khảo

In [21]:
answer_ex1_solution = 12.0

def triangle_area_exercise_solution(p, q, r):
    ux, uy = q[0] - p[0], q[1] - p[1]
    vx, vy = r[0] - p[0], r[1] - p[1]
    return abs(ux * vy - uy * vx) / 2

def is_boomerang_solution(points):
    return cross(*points) != 0

def count_positive_triangles_solution(points):
    return sum(cross(p, q, r) != 0 for p, q, r in combinations(points, 3))

def largest_triangle_vertices_solution(points):
    return max(combinations(points, 3), key=lambda t: triangle_area_exercise_solution(*t))

assert math.isclose(area_shoelace([1, 2], [4, 6], [7, 2]), answer_ex1_solution)
assert triangle_area_exercise_solution([0, 0], [0, 2], [2, 0]) == 2.0
assert is_boomerang_solution([[1, 1], [2, 3], [3, 2]]) is True
assert is_boomerang_solution([[1, 1], [2, 2], [3, 3]]) is False
assert is_boomerang_solution([[0, 0], [0, 0], [1, 1]]) is False      # điểm trùng
assert count_positive_triangles_solution(points1) == 8
assert largest_triangle_vertices_solution(points1) == ([0, 0], [0, 2], [2, 0])
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 17. Template nhớ nhanh và checklist submit

```python
import itertools

class Solution:
    def largestTriangleArea(self, points: list[list[int]]) -> float:
        best2 = 0
        for (x1, y1), (x2, y2), (x3, y3) in itertools.combinations(points, 3):
            best2 = max(best2, abs(x1*(y2-y3) + x2*(y3-y1) + x3*(y1-y2)))
        return best2 / 2
```

Công thức nhớ: **`2S = |(B−A) × (C−A)| = |x1(y2−y3) + x2(y3−y1) + x3(y1−y2)|`**.

**Checklist trước khi submit:**
- [ ] Có `abs(...)`?
- [ ] Chia `/ 2` (chia thực), không phải `// 2`?
- [ ] Duyệt `i < j < k` hoặc `combinations(points, 3)` (không chọn trùng điểm)?
- [ ] Trả kiểu `float`?
- [ ] Biết giải thích vì sao `O(n³)` đủ với `n ≤ 50`, và hướng bao lồi khi `n` lớn.